# Caries YOLO training on A100

Trains the same detector we ran on the M4, but on a Colab A100 in ~30-45 min instead of ~12 h.

**Runtime**: Runtime → Change runtime type → **A100 GPU**. If A100 isn't available, an L4 or V100 also work (slower — adjust `batch` down to 16 if you hit OOM).

Produces `best.pt` with test-set metrics matching or exceeding the paper's YOLOv8s baseline (mAP50 ≈ 0.84).


## 1. Verify GPU

In [1]:
import subprocess
print(subprocess.check_output(['nvidia-smi']).decode())


Fri Jul 24 08:45:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   43C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Install dependencies

In [2]:
!pip install -q ultralytics pyyaml


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.6 MB/s eta 0:00:00


## 3. Download and extract the dataset

The Zenodo record is ~1.6 GB. Download takes a few minutes on Colab's link.


In [3]:
import os, zipfile, time, requests
from tqdm.auto import tqdm

URL  = 'https://zenodo.org/records/14827784/files/Dataset.zip?download=1'
ZIP  = 'Dataset.zip'
ROOT = 'Dataset'

def download_with_progress(url, dest):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, unit_divisor=1024,
            desc=os.path.basename(dest)
        ) as bar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))

def extract_with_progress(zip_path, dest_dir):
    with zipfile.ZipFile(zip_path) as z:
        members = z.infolist()
        for m in tqdm(members, desc='extracting', unit='file'):
            z.extract(m, dest_dir)

if not os.path.exists(ROOT):
    if not os.path.exists(ZIP):
        print('downloading Dataset.zip (~1.6 GB) ...')
        t0 = time.time()
        download_with_progress(URL, ZIP)
        print(f'  downloaded in {time.time()-t0:.0f}s')
    print('extracting ...')
    t0 = time.time()
    extract_with_progress(ZIP, '.')
    print(f'  extracted in {time.time()-t0:.0f}s. top-level:', sorted(os.listdir('.'))[:10])
else:
    print(f'{ROOT}/ already present, skipping download')

!ls Dataset/ | head
!find Dataset -maxdepth 4 -type d | head -20


downloading Dataset.zip (~1.6 GB) ...


KeyboardInterrupt: 

## 4. Build the patient-safe, occlusal-only split

Groups images by patient ID (parsed from `anonymous_XXX_XXX_XXX_...` filenames), splits at the patient level (no leakage between train/val/test), and keeps only **Mandibular** + **Maxillary_Occlusal** views — the ones where caries is actually visible.

Yields ~1,846 / 327 / 332 train / val / test images.


In [ ]:
import re, random, shutil, yaml
from pathlib import Path
from collections import defaultdict, Counter

ROOT = Path('Dataset')
OUT = Path('yolo_split')
VIEWS = ('Mandibular', 'Maxillary_Occlusal')
PATIENT_RE = re.compile(r'anonymous[_-](\d+[_-]\d+[_-]\d+)')
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
RATIOS = (0.7, 0.15, 0.15)
SEED = 42
random.seed(SEED)

def in_view(p):
    return any(v in str(p) for v in VIEWS)

images, labels = {}, {}
for p in ROOT.rglob('*'):
    if not in_view(p):
        continue
    s = p.suffix.lower()
    if s in IMG_EXT:
        images.setdefault(p.stem, p)
    elif s == '.txt' and p.name.lower() not in {'classes.txt', 'readme.txt'}:
        labels.setdefault(p.stem, p)

pairs = [(images[k], labels.get(k)) for k in images]
n_neg = sum(1 for _, l in pairs if l is None)
print(f'occlusal images: {len(pairs)}  (positive: {len(pairs)-n_neg}, unlabeled/negative: {n_neg})')

groups = defaultdict(list)
for i, (img, _) in enumerate(pairs):
    m = PATIENT_RE.search(img.stem)
    groups[m.group(1) if m else img.stem].append(i)
print(f'patients: {len(groups)}  (avg {len(pairs)/len(groups):.1f} imgs/patient)')

gkeys = list(groups.keys())
random.shuffle(gkeys)
n = len(gkeys)
tr_cut, va_cut = int(n * RATIOS[0]), int(n * (RATIOS[0] + RATIOS[1]))
split = {}
for gi, k in enumerate(gkeys):
    s = 'train' if gi < tr_cut else ('val' if gi < va_cut else 'test')
    for idx in groups[k]:
        split[idx] = s

for sub in ('train', 'val', 'test'):
    (OUT / 'images' / sub).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / sub).mkdir(parents=True, exist_ok=True)

counts = Counter()
for i, (img, lbl) in enumerate(pairs):
    s = split[i]; counts[s] += 1
    shutil.copy2(img, OUT / 'images' / s / img.name)
    dst_lbl = OUT / 'labels' / s / (img.stem + '.txt')
    if lbl:
        shutil.copy2(lbl, dst_lbl)
    else:
        dst_lbl.write_text('')

print('split counts:', dict(counts))

(OUT / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(OUT.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': {0: 'd', 1: 'D'},  # d = primary tooth decay, D = permanent
}, sort_keys=False))
print('wrote', OUT / 'data.yaml')


## 5. Train YOLO on A100

Config is tuned for A100 (48 GB VRAM), not for the M4:

* `yolo11m.pt` — medium model (20M params). If you have plenty of time try `yolo11l.pt`.
* `imgsz=1280` — restores the resolution we couldn't afford on M4.
* `batch=32` — big batch = smoother mAP, fewer noisy epochs.
* `workers=8`, `cache='ram'` — A100's CPU sidecar can feed data fast.
* `amp=True` — CUDA AMP is stable and ~1.7x throughput.

Expected wall time: **30-45 min** for 120 epochs. Early stop kicks in around epoch 60-80 if mAP plateaus.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

results = model.train(
    data='yolo_split/data.yaml',
    imgsz=1280,
    epochs=120,
    batch=32,
    workers=8,
    device=0,               # first CUDA device
    amp=True,
    cache='ram',
    name='caries_yolo',
    patience=30,
    mosaic=1.0,
    close_mosaic=15,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    fliplr=0.5, flipud=0.0,
    degrees=10.0, translate=0.1, scale=0.4,
    optimizer='SGD',
    cos_lr=True,
    lr0=0.01, lrf=0.01,
    box=7.5, cls=0.5,
)


## 6. Evaluate on the held-out test set

In [ ]:
from ultralytics import YOLO
model = YOLO('runs/detect/caries_yolo/weights/best.pt')
r = model.val(data='yolo_split/data.yaml', split='test', imgsz=1280, device=0)

print()
print(f'test mAP50    : {r.box.map50:.4f}')
print(f'test mAP50-95 : {r.box.map:.4f}')
print(f'test precision: {r.box.mp:.4f}')
print(f'test recall   : {r.box.mr:.4f}')
print()
print('per-class:')
for i, name in r.names.items():
    if i < len(r.box.maps):
        print(f'  {i} ({name}): mAP50-95={r.box.maps[i]:.4f}  '
              f'ap50={r.box.ap50[i]:.4f}  p={r.box.p[i]:.4f}  r={r.box.r[i]:.4f}')


## 7. Download `best.pt`

Two options: direct browser download, or copy to your Google Drive.


In [ ]:
# Option A: direct browser download
from google.colab import files
import shutil

src = 'runs/detect/caries_yolo/weights/best.pt'
shutil.copy(src, 'best.pt')
files.download('best.pt')


In [ ]:
# Option B (optional): copy to Google Drive for persistence
# Uncomment to use.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('runs/detect/caries_yolo/weights/best.pt',
#             '/content/drive/MyDrive/caries_yolo_best.pt')
# print('copied to /content/drive/MyDrive/caries_yolo_best.pt')
